# Exercise 2: Structured Streaming + Transformations

The objective of this exercise is to understand how to apply transformations of a DataFrame inside a Stream, such as:

- select
- withColumn
- filter
- current_timestamp
- col

In other words, we want to demonstrate that a stream can be transformed as a batch DataFrame, whenever the transformations are deterministic and supported by streaming.

## Step 1. Stream Creation

As in Exercise 1. We are going to create a stream, which will create a new schema to not run into the schema from Exercise 1, but will refer to the same input folder

In [0]:
from pyspark.sql.functions import col, current_timestamp

df_stream2 = (
    spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", "/Volumes/workspace/default/streaming_demo/schema2")
        .load("/Volumes/workspace/default/streaming_demo/input")
        .withColumn("processed_at", current_timestamp())
        .filter(col("id") > 0)
)


# Step 2: Stream Writing to Delta

Now we use the writeStream option, since the input folder inside the Volume is not empty this code will pass through directly.

In [0]:
(df_stream2.writeStream
    .format("delta")
    .option("checkpointLocation", "/Volumes/workspace/default/streaming_demo/chk2")
    .trigger(availableNow=True)
    .outputMode("append")
    .table("workspace.default.streaming_demo_transformed"))

# Step 3: New File Input to test the Stream

A new file is uploaded to input and then we run the writeStream command to update manually (In Databricks non-Free editions, when `.trigger` option is set to "auto", this step only is reduced to upload new files to the input)

In [0]:
dbutils.fs.put("/Volumes/workspace/default/streaming_demo/input/file10.json",
               """{"id": 10, "value": "archivo ejercicio 2"}""", True)

Wrote 42 bytes.


True

# Step 4: Validation

Now we validate that the new file was updated in the table

In [0]:
%sql
SELECT * FROM workspace.default.streaming_demo_transformed;

id,value,_rescued_data,processed_at
2,nuevo archivo 2,null,2026-04-16T13:52:20.353Z
3,nuevo archivo 3,null,2026-04-16T13:52:20.353Z
4,nuevo archivo 4,null,2026-04-16T13:52:20.353Z
5,nuevo archivo 5,null,2026-04-16T13:52:20.353Z
2,mundo,null,2026-04-16T13:52:20.353Z
10,archivo ejercicio 2,null,2026-04-16T13:56:09.437Z
